# Local TikZ n-gram repetition

This notebook detects repetition **inside each individual TikZ/LaTeX sample**.

It does **not** compare samples against the complete dataset.

Pipeline:

```text
one TikZ sample
→ remove comments
→ normalize numbers to NUM
→ tokenize
→ build 1-, 2-, 3-, 4-grams
→ count repeated n-grams inside this sample
→ compute repetition coverage
→ low / medium / high / critical
```

The original code is written into `low/`, `medium/`, or `critical/`. A CSV with metrics and the most frequent local n-grams is written as well.


In [1]:
from collections import Counter
from pathlib import Path
import json
import re
import shutil

import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm


/home/jonas/miniconda3/envs/data-tools/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [2]:
PARQUET_DIR = Path("/home/jonas/Datasets/TikZ/DatikZ-v4-raw")
OUTPUT_DIR = Path("./local_repetition_test")

CODE_COLUMNS = (
    "reference_code",
    "code",
    "tikz_code",
)

NGRAM_ORDERS = (1, 2, 3, 4)

NGRAM_WEIGHTS = {
    1: 0.40,
    2: 0.30,
    3: 0.20,
    4: 0.10,
}

# An n-gram must occur this often inside the SAME sample.
MIN_REPEAT_COUNT = 2

MEDIUM_THRESHOLD = 0.40
HIGH_THRESHOLD = 0.60
CRITICAL_THRESHOLD = 0.80

TOP_LOCAL_NGRAMS = 20
BATCH_SIZE = 2048
OVERWRITE = True


## Normalize and tokenize

In [3]:
NUMBER_RE = re.compile(
    r"""
    (?<![A-Za-z])
    [-+]?
    (?:
        \d+\.\d*
        |
        \.\d+
        |
        \d+
    )
    (?:[eE][-+]?\d+)?
    """,
    re.VERBOSE,
)

TOKEN_RE = re.compile(
    r"""
    \\[A-Za-z@]+
    | NUM
    | [A-Za-z_][A-Za-z0-9_]*
    | --|->|<-|<->
    | [^\s]
    """,
    re.VERBOSE,
)


def strip_comments(code: str) -> str:
    return "\n".join(
        re.sub(r"(?<!\\)%.*$", "", line)
        for line in str(code or "").splitlines()
    )


def normalize(code: str) -> str:
    return NUMBER_RE.sub("NUM", strip_comments(code))


def tokenize(code: str) -> list[str]:
    return TOKEN_RE.findall(normalize(code))


## Local n-gram repetition

In [4]:
def make_ngrams(tokens: list[str], n: int) -> list[tuple[str, ...]]:
    return [
        tuple(tokens[i:i + n])
        for i in range(len(tokens) - n + 1)
    ]


def local_ngram_metrics(tokens: list[str], n: int) -> dict:
    grams = make_ngrams(tokens, n)

    if not grams:
        return {
            "coverage": 0.0,
            "max_frequency": 0,
            "counts": Counter(),
        }

    counts = Counter(grams)
    repeated_occurrences = sum(
        count
        for count in counts.values()
        if count >= MIN_REPEAT_COUNT
    )

    return {
        "coverage": repeated_occurrences / len(grams),
        "max_frequency": max(counts.values()),
        "counts": counts,
    }


def repetition_metrics(code: str) -> dict:
    tokens = tokenize(code)

    if not tokens:
        return {
            "repetition_score": 0.0,
            "repetition_class": "low",
            "token_count": 0,
            **{f"ngram_{n}_coverage": 0.0 for n in NGRAM_ORDERS},
            "top_local_ngrams": "[]",
        }

    score = 0.0
    output = {}
    repeated = []

    for n in NGRAM_ORDERS:
        metrics = local_ngram_metrics(tokens, n)
        coverage = metrics["coverage"]
        score += NGRAM_WEIGHTS[n] * coverage
        output[f"ngram_{n}_coverage"] = coverage

        for gram, count in metrics["counts"].most_common():
            if count < MIN_REPEAT_COUNT:
                break
            repeated.append({
                "n": n,
                "count": count,
                "ngram": " ".join(gram),
            })

    repeated.sort(key=lambda x: (x["count"], x["n"]), reverse=True)

    if score < MEDIUM_THRESHOLD:
        repetition_class = "low"
    elif score < HIGH_THRESHOLD:
        repetition_class = "medium"
    elif score < CRITICAL_THRESHOLD:
        repetition_class = "high"
    else:
        repetition_class = "critical"

    return {
        "repetition_score": score,
        "repetition_class": repetition_class,
        "token_count": len(tokens),
        **output,
        "top_local_ngrams": json.dumps(
            repeated[:TOP_LOCAL_NGRAMS],
            ensure_ascii=False,
        ),
    }


## Quick test

In [5]:
example = r"""
\begin{tikzpicture}
    \draw (0,0) -- (1,1);
    \draw (1,1) -- (2,2);
    \draw (2,2) -- (3,3);
    \draw (3,3) -- (4,4);
    \draw (4,4) -- (5,5);
    \draw (5,5) -- (6,6);
\end{tikzpicture}
"""

display(pd.DataFrame([repetition_metrics(example)]))


,repetition_score,repetition_class,token_count,ngram_1_coverage,ngram_2_coverage,ngram_3_coverage,ngram_4_coverage,top_local_ngrams
0,0.952656,critical,86,0.976744,0.952941,0.928571,0.903614,"[{""n"": 1, ""count"": 24, ""ngram"": ""NUM""}, {""n"": ..."


In [6]:
def show_local_ngrams(code: str, n: int, top_k: int = 30):
    counts = Counter(make_ngrams(tokenize(code), n))
    display(pd.DataFrame([
        {"count": count, "ngram": " ".join(gram)}
        for gram, count in counts.most_common(top_k)
    ]))

show_local_ngrams(example, n=2)


,count,ngram
0,12,( NUM
1,12,"NUM ,"
2,12,", NUM"
3,12,NUM )
4,6,\draw (
5,6,) --
6,6,-- (
7,6,) ;
8,5,; \draw
9,2,{ tikzpicture


## Process Parquet files

In [7]:
def find_code_column(parquet_file: Path) -> str:
    columns = set(pq.ParquetFile(parquet_file).schema.names)

    for column in CODE_COLUMNS:
        if column in columns:
            return column

    raise ValueError(
        f"No code column found in {parquet_file}. Tried: {CODE_COLUMNS}"
    )


def prepare_output():
    if OUTPUT_DIR.exists():
        if not OVERWRITE:
            raise FileExistsError(
                f"{OUTPUT_DIR} already exists. Set OVERWRITE=True to replace it."
            )
        shutil.rmtree(OUTPUT_DIR)

    for class_name in ("low", "medium", "high", "critical"):
        (OUTPUT_DIR / class_name).mkdir(parents=True, exist_ok=True)


parquet_files = sorted(PARQUET_DIR.rglob("*.parquet"))

if not parquet_files:
    raise FileNotFoundError(f"No Parquet files found in {PARQUET_DIR}")

prepare_output()

rows = []
sample_index = 0

for parquet_file in tqdm(parquet_files, desc="Parquet files"):
    code_column = find_code_column(parquet_file)
    parquet = pq.ParquetFile(parquet_file)

    for batch in parquet.iter_batches(
        columns=[code_column],
        batch_size=BATCH_SIZE,
    ):
        for code in batch.column(0).to_pylist():
            code = str(code or "")
            metrics = repetition_metrics(code)
            class_name = metrics["repetition_class"]

            output_file = (
                OUTPUT_DIR
                / class_name
                / f"{sample_index:09d}.txt"
            )
            output_file.write_text(code, encoding="utf-8")

            rows.append({
                "sample_index": sample_index,
                "source_file": str(parquet_file),
                "output_file": str(output_file),
                **metrics,
            })
            sample_index += 1

results = pd.DataFrame(rows)
results.to_csv(
    OUTPUT_DIR / "local_repetition_results.csv",
    index=False,
)

print(f"Processed: {len(results):,}")
display(
    results["repetition_class"]
    .value_counts()
    .reindex(["low", "medium", "high", "critical"], fill_value=0)
    .to_frame("samples")
)


Parquet files: 100%|██████████| 86/86 [02:48<00:00,  1.96s/it]


Processed: 427,753


,samples
repetition_class,
low,2624
medium,75686
high,177884
critical,171559


## Inspect result distribution

In [8]:
display(
    results.groupby("repetition_class")[[
        "repetition_score",
        "ngram_1_coverage",
        "ngram_2_coverage",
        "ngram_3_coverage",
        "ngram_4_coverage",
    ]].describe()
)

display(
    results.sort_values(
        "repetition_score",
        ascending=False,
    ).head(50)
)


repetition_score                                          \
                            count      mean       std       min       25%   
repetition_class                                                            
critical                 171559.0  0.880696  0.051497  0.800001  0.837090   
high                     177884.0  0.711830  0.056445  0.600002  0.666040   
low                        2624.0  0.377567  0.023929  0.216089  0.370795   
medium                    75686.0  0.519394  0.053368  0.400159  0.477567   

                                               ngram_1_coverage            \
                       50%       75%       max            count      mean   
repetition_class                                                            
critical          0.875873  0.918631  0.999092         171559.0  0.956983   
high              0.717458  0.761051  0.799999         177884.0  0.889058   
low               0.384032  0.393359  0.399791           2624.0  0.675633   
medium            0.525811  0.565682  0.599998          75686.0  0.791514   

                  ... ngram_3_coverage           ngram_4_coverage            \
                  ...              75%       max            count      mean   
repetition_class  ...                                                         
critical          ...         0.866477  0.998380         171559.0  0.735072   
high              ...         0.608187  0.722826         177884.0  0.407422   
low               ...         0.125000  0.225000           2624.0  0.004883   
medium            ...         0.313725  0.455157          75686.0  0.110993   

                                                                              
                       std       min       25%       50%       75%       max  
repetition_class                                                              
critical          0.120130  0.372093  0.642527  0.726914  0.823570  0.997840  
high              0.109729  0.000000  0.323699  0.412371  0.493274  0.693989  
low               0.012178  0.000000  0.000000  0.000000  0.000000  0.153846  
medium            0.075591  0.000000  0.049035  0.104575  0.167785  0.393617  

[4 rows x 40 columns]

,sample_index,source_file,output_file,repetition_score,repetition_class,token_count,ngram_1_coverage,ngram_2_coverage,ngram_3_coverage,ngram_4_coverage,top_local_ngrams
88286,88286,/home/jonas/Datasets/TikZ/DatikZ-v4-raw/train-...,local_repetition_test/critical/000088286.txt,0.999092,critical,23120,0.999784,0.999092,0.998356,0.997794,"[{""n"": 1, ""count"": 5766, ""ngram"": ""NUM""}, {""n""..."
19926,19926,/home/jonas/Datasets/TikZ/DatikZ-v4-raw/train-...,local_repetition_test/critical/000019926.txt,0.999019,critical,21613,0.999630,0.999028,0.998380,0.997825,"[{""n"": 1, ""count"": 5525, ""ngram"": ""NUM""}, {""n""..."
166027,166027,/home/jonas/Datasets/TikZ/DatikZ-v4-raw/train-...,local_repetition_test/critical/000166027.txt,0.998977,critical,24729,0.999636,0.998989,0.998261,0.997735,"[{""n"": 1, ""count"": 5997, ""ngram"": ""NUM""}, {""n""..."
130115,130115,/home/jonas/Datasets/TikZ/DatikZ-v4-raw/train-...,local_repetition_test/critical/000130115.txt,0.998942,critical,22869,0.999694,0.999038,0.998076,0.997376,"[{""n"": 1, ""count"": 7019, ""ngram"": ""NUM""}, {""n""..."
220343,220343,/home/jonas/Datasets/TikZ/DatikZ-v4-raw/train-...,local_repetition_test/critical/000220343.txt,0.998856,critical,32866,0.999483,0.998752,0.998266,0.997840,"[{""n"": 1, ""count"": 8198, ""ngram"": ""NUM""}, {""n""..."
130114,130114,/home/jonas/Datasets/TikZ/DatikZ-v4-raw/train-...,local_repetition_test/critical/000130114.txt,0.998842,critical,20893,0.999665,0.998947,0.997894,0.997128,"[{""n"": 1, ""count"": 6411, ""ngram"": ""NUM""}, {""n""..."
220368,220368,/home/jonas/Datasets/TikZ/DatikZ-v4-raw/train-...,local_repetition_test/critical/000220368.txt,0.998783,critical,32882,0.999483,0.998753,0.998084,0.997476,"[{""n"": 1, ""count"": 8198, ""ngram"": ""NUM""}, {""n""..."
166029,166029,/home/jonas/Datasets/TikZ/DatikZ-v4-raw/train-...,local_repetition_test/critical/000166029.txt,0.998743,critical,20126,0.999553,0.998758,0.997863,0.997217,"[{""n"": 1, ""count"": 4895, ""ngram"": ""NUM""}, {""n""..."
220344,220344,/home/jonas/Datasets/TikZ/DatikZ-v4-raw/train-...,local_repetition_test/critical/000220344.txt,0.998657,critical,32908,0.999423,0.998541,0.997934,0.997386,"[{""n"": 1, ""count"": 8205, ""ngram"": ""NUM""}, {""n""..."
63019,63019,/home/jonas/Datasets/TikZ/DatikZ-v4-raw/train-...,local_repetition_test/critical/000063019.txt,0.998641,critical,21787,0.999449,0.998623,0.997843,0.997062,"[{""n"": 1, ""count"": 6056, ""ngram"": ""NUM""}, {""n""..."
